# Dataset Analytics Cookbook: Interactive RAG Workflow

This notebook demonstrates an end-to-end RAG pipeline for dataset analysis.

**Workflow:**
1. Load public Titanic dataset
2. Initialize RAG engine with local deterministic embeddings
3. Query and retrieve grounded results
4. Generate answers with citations
5. Run evaluation gates and inspect the summary

**Time:** ~10 minutes | **Prerequisites:** Python 3.9+, ragsearch library

## Setup: Import Libraries and Initialize

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Ensure ragsearch imports work
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "libs").exists():
        ROOT = candidate
        break
LIBS_ROOT = ROOT / "libs"
if str(LIBS_ROOT) not in sys.path:
    sys.path.insert(0, str(LIBS_ROOT))

from ragsearch.engine import RagSearchEngine
from ragsearch.vector_db import VectorDB
from ragsearch.evaluation import run_regression_gates, EvaluationThresholds

print("✓ Imports successful")

## Step 1: Define Deterministic Embedding and LLM Models

In [ ]:
class DummyEmbeddingResponse:
    def __init__(self, embeddings):
        self.embeddings = embeddings

class KeywordEmbeddingModel:
    """Keyword-based embedding for reproducible demo.
    
    In production, use real embeddings:
    - Cohere: create_embedding_model('cohere', api_key=...)
    - OpenAI: create_embedding_model('openai', api_key=...)
    - Local: create_embedding_model('sentence_transformers')
    """
    def embed(self, texts):
        vectors = []
        for text in texts:
            lowered = str(text).lower()
            # Create 4D vectors based on keyword presence
            v = [0.1, 0.1, 0.1, 0.1]
            if "female" in lowered or "woman" in lowered:
                v[0] += 1.0
            if "male" in lowered or "man" in lowered:
                v[1] += 1.0
            if "first" in lowered or "class 1" in lowered:
                v[2] += 1.0
            if "third" in lowered or "class 3" in lowered:
                v[3] += 1.0
            vectors.append(v)
        return DummyEmbeddingResponse(vectors)

class DummyLLMClient:
    """Placeholder LLM for demo.
    
    In production, use real providers:
    - Cohere: create_llm_client('cohere', api_key=...)
    - OpenAI: create_llm_client('openai', api_key=...)
    - Ollama: create_llm_client('ollama', base_url=...)
    """
    def generate(self, prompt, **kwargs):
        return "Answer generated from retrieved Titanic passenger records with citations."

print("✓ Embedding and LLM models defined")

## Step 2: Load and Normalize Titanic Dataset

In [ ]:
print("Loading public Titanic dataset...")
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_raw = pd.read_csv(DATA_URL)

# Use first 100 rows for faster execution
df = df_raw.head(100).copy()

print(f"✓ Loaded {len(df)} passenger records")
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:5]}...")
print(f"\nFirst record:\n{df.iloc[0][:5]}")

## Step 3: Build Text Field and Prepare for Indexing

In [ ]:
def build_text_field(row):
    """Combine relevant columns into single text field for embedding."""
    pieces = []
    for col in ["Name", "Sex", "Age", "Pclass", "Survived", "Fare"]:
        if col in row and pd.notna(row[col]):
            pieces.append(f"{col}: {row[col]}")
    return " | ".join(pieces)

df["text"] = df.apply(build_text_field, axis=1)
df["source_path"] = "public:titanic.csv"
df["parser_name"] = "public/csv"

print("✓ Text field created")
print(f"\nExample text field:\n{df['text'].iloc[0]}")

## Step 4: Initialize RAG Engine and Index Data

In [ ]:
print("Initializing RAG Engine...")
engine = RagSearchEngine(
    data=df,
    embedding_model=KeywordEmbeddingModel(),
    llm_client=DummyLLMClient(),
    vector_db=VectorDB(embedding_dim=4),
    save_dir="notebook_embeddings",
    file_name="titanic.csv",
)

print("✓ Engine initialized and data indexed")

# Display indexing diagnostics
diag = engine.indexing_diagnostics
print(f"\nIndexing Diagnostics:")
print(f"  Total records: {diag['total_records']}")
print(f"  Embedded: {diag['embedded_records']}")
print(f"  New: {diag['new_records']}")
print(f"  Reused (cached): {diag['reused_records']}")

## Step 5: Query and Retrieve Results

In [ ]:
# Example query 1
query_1 = "female passengers who survived"
print(f"Query: {query_1}")
results_1 = engine.search(query_1, top_k=3)

print(f"\nRetrieved {len(results_1)} results:")
for i, result in enumerate(results_1, start=1):
    citation = result["citation"]
    print(f"\n[{i}] Similarity: {result['similarity']:.3f}")
    print(f"    Source: {citation['source_path']}")
    print(f"    Excerpt: {citation['excerpt'][:75]}...")

## Step 6: Generate Grounded Answers

In [ ]:
# Generate answer with citations
query_2 = "What were the characteristics of third-class male passengers?"
print(f"Query: {query_2}\n")

answer = engine.answer(query_2, top_k=3)

print(f"Answer:")
print(answer["answer"])

print(f"\n\nCitations ({len(answer['citations'])} sources):")
for i, cit in enumerate(answer["citations"], start=1):
    print(f"\n[{i}] {cit['source_path']}")
    print(f"    {cit['excerpt'][:80]}...")

## Step 7: Run Evaluation Gates and Track Metrics

In [ ]:
# Define test cases
test_cases = [
    {"query": "female passengers", "top_k": 5, "min_results": 1, "min_citations": 1},
    {"query": "first class passengers", "top_k": 5, "min_results": 1, "min_citations": 1},
    {"query": "survival rates", "top_k": 5, "min_results": 1, "min_citations": 1},
]

print("Running evaluation gates...\n")
summary = run_regression_gates(
    engine,
    test_cases,
    EvaluationThresholds(min_results=1, min_citations=1)
)

print(f"Result: {'✓ PASS' if summary['pass'] else '✗ FAIL'}")
print(f"Passed: {summary['passed_cases']}/{summary['total_cases']}\n")

for result in summary["results"]:
    status = "✓" if result["passed"] else "✗"
    print(f"{status} {result['query']}")
    print(f"   Results: {result['observed_results']}/{result['expected_min_results']}, "
          f"Citations: {result['observed_citations']}/{result['expected_min_citations']}")

## Step 8: Access Observability Events

In [ ]:
# View low-level observability events
print("Recent observability events:\n")
for event in engine.observability_events[-3:]:
    print(f"{event['stage']}: {event['event']}")
    payload = event['payload']
    if 'latency_ms' in payload:
        print(f"  Latency: {payload['latency_ms']}ms")
    if 'results_count' in payload:
        print(f"  Results: {payload['results_count']}")
    print()

## Summary

**This notebook demonstrated:**
- ✓ Loading and preparing structured data
- ✓ Initializing a deterministic RAG engine
- ✓ Retrieving grounded results with citations
- ✓ Generating answers backed by sources
- ✓ Running evaluation gates
- ✓ Accessing observability events

**Next steps:**
1. Replace dummy embedding/LLM with real providers (Cohere, OpenAI, etc)
2. Expand to larger datasets and production queries
3. Use your own runner to persist benchmark runs under `.benchmarks/` if you want trend tracking
4. See [Troubleshooting](./troubleshooting.md) for common issues
5. See [Reference](./reference-api-cheat-sheet.md) for full API documentation